In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/20260326 - List of quotes.csv')
df.head()

import pandas as pd
import numpy as np

print("="*80)
print("DATA CLEANING & PREPARATION")
print("="*80)

# Assuming df is already loaded from your CSV
print(f"Initial data loaded. Shape: {df.shape}")
print(f"Columns: {list(df.columns)[:10]}...")  # Show first 10 columns

# 1. Identify test accounts
print("\n🔍 IDENTIFYING TEST/TRAINING ACCOUNTS...")

def identify_test_accounts(df):
    """Identify suspicious accounts for removal"""
    
    # Known test account from discovery
    known_test = ['CL00201682']
    
    # Pattern 1: High volume, zero conversion
    print("  Analyzing high-volume, zero-conversion accounts...")
    customer_stats = df.groupby('numero_compte').agg({
        'fg_devis_accepte': ['count', 'mean']
    })
    customer_stats.columns = ['quote_count', 'conversion_rate']
    
    # Find accounts with >20 quotes and 0% conversion
    high_volume_zero = customer_stats[
        (customer_stats['quote_count'] > 20) & 
        (customer_stats['conversion_rate'] == 0)
    ].sort_values('quote_count', ascending=False)
    
    print(f"  Found {len(high_volume_zero)} accounts with >20 quotes and 0% conversion")
    if len(high_volume_zero) > 0:
        print(f"  Top 5 by volume: {high_volume_zero.index[:5].tolist()}")
    
    # Pattern 2: Unrealistic quote frequencies (>1 quote per day average)
    print("\n  Analyzing quote frequency patterns...")
    df['dt_creation_devis'] = pd.to_datetime(df['dt_creation_devis'])
    
    customer_timelines = df.groupby('numero_compte').agg({
        'dt_creation_devis': ['min', 'max', 'count']
    })
    customer_timelines.columns = ['first_quote', 'last_quote', 'quote_count']
    
    customer_timelines['days_active'] = (
        customer_timelines['last_quote'] - customer_timelines['first_quote']
    ).dt.days + 1
    
    # Avoid division by zero for single-quote customers
    customer_timelines['quotes_per_day'] = np.where(
        customer_timelines['days_active'] > 0,
        customer_timelines['quote_count'] / customer_timelines['days_active'],
        1  # Single quote on one day = 1 quote/day
    )
    
    unrealistic_frequency = customer_timelines[
        customer_timelines['quotes_per_day'] > 1
    ].sort_values('quotes_per_day', ascending=False)
    
    print(f"  Found {len(unrealistic_frequency)} accounts with >1 quote/day average")
    if len(unrealistic_frequency) > 0:
        print(f"  Most extreme: {unrealistic_frequency.index[0]} with {unrealistic_frequency.iloc[0]['quotes_per_day']:.1f} quotes/day")
    
    # Pattern 3: Employees/Internal (simplified - would need HR data in reality)
    print("\n  Checking for potential employee accounts...")
    # This is a placeholder - in reality would cross-reference with HR database
    # For now, look for patterns like company email domains, known employee names
    employee_patterns = []
    
    # Combine all suspicious accounts
    all_suspicious = list(set(
        known_test + 
        high_volume_zero.index.tolist()[:10] +  # Top 10 high-volume, zero-conversion
        unrealistic_frequency.index.tolist()[:5]  # Top 5 by frequency
    ))
    
    return all_suspicious, high_volume_zero, unrealistic_frequency

# Run test account identification
test_accounts, high_volume_zero, unrealistic_frequency = identify_test_accounts(df)

print(f"\n📊 TEST ACCOUNT ANALYSIS:")
print(f"Total suspicious accounts identified: {len(test_accounts)}")
print(f"Known test account (CL00201682): 229 quotes, 0% conversion")

# Show impact of each pattern
if len(high_volume_zero) > 0:
    top_high_volume = high_volume_zero.iloc[0]
    print(f"Highest volume zero-conversion: {high_volume_zero.index[0]} with {int(top_high_volume['quote_count'])} quotes")

if len(unrealistic_frequency) > 0:
    top_frequency = unrealistic_frequency.iloc[0]
    print(f"Highest frequency: {unrealistic_frequency.index[0]} with {top_frequency['quotes_per_day']:.1f} quotes/day")

# 2. Remove test accounts and create clean dataset
print("\n🧹 CREATING CLEAN DATASET...")
df_clean = df[~df['numero_compte'].isin(test_accounts)].copy()
quotes_removed = len(df) - len(df_clean)
percent_removed = (quotes_removed / len(df)) * 100

print(f"Original dataset: {len(df):,} quotes")
print(f"Clean dataset: {len(df_clean):,} quotes")
print(f"Quotes removed: {quotes_removed:,} ({percent_removed:.1f}%)")

# 3. Calculate baseline statistics
print("\n📈 BASELINE STATISTICS (CLEAN DATA):")

# Basic metrics
total_customers = df_clean['numero_compte'].nunique()
total_quotes = len(df_clean)
quotes_per_customer = total_quotes / total_customers
conversion_rate = df_clean['fg_devis_accepte'].mean()

print(f"Total unique customers: {total_customers:,}")
print(f"Total quotes: {total_quotes:,}")
print(f"Avg quotes per customer: {quotes_per_customer:.2f}")
print(f"Quote-level conversion rate: {conversion_rate:.1%}")

# Customer-level conversion
customer_conversion = df_clean.groupby('numero_compte')['fg_devis_accepte'].max()
customer_conversion_rate = customer_conversion.mean()
customers_with_sales = customer_conversion.sum()

print(f"\nCustomer-level metrics:")
print(f"Customers with at least one sale: {customers_with_sales:,} ({customer_conversion_rate:.1%})")
print(f"Customers with no sales: {total_customers - customers_with_sales:,} ({1 - customer_conversion_rate:.1%})")

# Distribution analysis
print("\n📊 DISTRIBUTION ANALYSIS:")
quote_counts = df_clean['numero_compte'].value_counts()
print(f"Customers with 1 quote: {(quote_counts == 1).sum():,} ({(quote_counts == 1).sum()/total_customers:.1%})")
print(f"Customers with 2-3 quotes: {((quote_counts >= 2) & (quote_counts <= 3)).sum():,} ({((quote_counts >= 2) & (quote_counts <= 3)).sum()/total_customers:.1%})")
print(f"Customers with 4+ quotes: {(quote_counts >= 4).sum():,} ({(quote_counts >= 4).sum()/total_customers:.1%})")

# Temporal analysis
print("\n📅 TEMPORAL ANALYSIS:")
df_clean['dt_creation_devis'] = pd.to_datetime(df_clean['dt_creation_devis'])
print(f"Date range: {df_clean['dt_creation_devis'].min().date()} to {df_clean['dt_creation_devis'].max().date()}")

# Check data quality issues
print("\n🔍 DATA QUALITY CHECK:")
missing_target = df_clean['fg_devis_accepte'].isna().sum()
print(f"Missing target values: {missing_target} ({missing_target/len(df_clean):.1%})")

# Remove any remaining NaN in target (if any)
if missing_target > 0:
    df_clean = df_clean.dropna(subset=['fg_devis_accepte'])
    print(f"Removed {missing_target} rows with NaN target")

# 4. Save clean dataset for Day 2
output_file = 'cleaned_quote_data.csv'
df_clean.to_csv(output_file, index=False)
print(f"\n💾 Clean dataset saved to: {output_file}")


DATA CLEANING & PREPARATION
Initial data loaded. Shape: (37371, 50)
Columns: ['id_devis', 'num_devis', 'nom_devis', 'nom_agence', 'nom_filiale_zone', 'nom_region', 'statut_devis', 'fg_devis_emis', 'fg_devis_refuse', 'fg_devis_accepte']...

🔍 IDENTIFYING TEST/TRAINING ACCOUNTS...
  Analyzing high-volume, zero-conversion accounts...
  Found 3 accounts with >20 quotes and 0% conversion
  Top 5 by volume: ['CL00201682', 'CL00069020', 'CL00294368']

  Analyzing quote frequency patterns...
  Found 3729 accounts with >1 quote/day average
  Most extreme: CL00338101 with 6.0 quotes/day

  Checking for potential employee accounts...

📊 TEST ACCOUNT ANALYSIS:
Total suspicious accounts identified: 8
Known test account (CL00201682): 229 quotes, 0% conversion
Highest volume zero-conversion: CL00201682 with 263 quotes
Highest frequency: CL00338101 with 6.0 quotes/day

🧹 CREATING CLEAN DATASET...
Original dataset: 37,371 quotes
Clean dataset: 37,015 quotes
Quotes removed: 356 (1.0%)

📈 BASELINE STATIS

In [2]:
list(df_clean.columns)

['id_devis',
 'num_devis',
 'nom_devis',
 'nom_agence',
 'nom_filiale_zone',
 'nom_region',
 'statut_devis',
 'fg_devis_emis',
 'fg_devis_refuse',
 'fg_devis_accepte',
 'dt_creation_devis',
 'dt_signature_devis',
 'fg_3_mois_mature',
 'type_devis',
 'mt_apres_remise_ht_devis',
 'mt_marge',
 'nb_devis_emis',
 'mt_apres_remise_ht_emis_devis',
 'mt_marge_emis_devis',
 'mt_remise_exceptionnelle_ht',
 'mt_ttc_apres_aide_devis',
 'mt_ttc_avant_aide_devis',
 'mt_prime_cee',
 'mt_prime_maprimerenov',
 'fg_activite_commerciale',
 'prenom_nom_createur',
 'prenom_nom_commercial',
 'nom_campagne',
 'famille_equipement_produit',
 'type_equipement_produit',
 'dth_emission_devis',
 'dt_emission_calcule_devis',
 'id_opportunite',
 'fg_devis_principal',
 'lb_statut_preparation_chantier',
 'numero_compte',
 'dt_visite_commerciale',
 'fonction_commercial',
 'fonction_createur',
 'regroup_famille_equipement_produit',
 'marque_produit',
 'modele_produit',
 'famille_equipement_produit_principal',
 'regroup_

In [3]:
# HEAT PUMP OWNERS BY REGION (UPDATED)
print("\n" + "="*80)
print("HEAT PUMP OWNERS BY REGION")
print("="*80)

# Load data
df_quotes = pd.read_csv('cleaned_quote_data.csv')
df_quotes['dt_creation_devis'] = pd.to_datetime(df_quotes['dt_creation_devis'])

# Get customers with heat pump
has_heat_pump = df_quotes.groupby('numero_compte')['famille_equipement_produit'].apply(
    lambda x: 'Pompe à chaleur' in x.values
)

# Calculate by region
heat_pump_by_region = df_quotes[df_quotes['numero_compte'].isin(has_heat_pump[has_heat_pump].index)] \
    .groupby('nom_region')['numero_compte'].nunique() \
    .reset_index(name='heat_pump_owners')

# Total customers by region
total_by_region = df_quotes.groupby('nom_region')['numero_compte'].nunique().reset_index(name='total_customers')

# Merge
region_stats = heat_pump_by_region.merge(total_by_region, on='nom_region', how='right').fillna(0)
region_stats['pct_of_region'] = (region_stats['heat_pump_owners'] / region_stats['total_customers'] * 100).round(1)

# Sort and display
region_stats = region_stats.sort_values('heat_pump_owners', ascending=False)
print(region_stats[['nom_region', 'heat_pump_owners', 'pct_of_region']].to_string(index=False))

print("\nCold regions have more heat pumps than warm regions.")


HEAT PUMP OWNERS BY REGION
          nom_region  heat_pump_owners  pct_of_region
           Normandie              3499           31.8
Auvergne-Rhône-Alpes              1101           19.3
     Hauts-de-France               272            7.5
       Île-de-France               150            3.2
                 Sud                56            8.7

Cold regions have more heat pumps than warm regions.


In [4]:
# SINGLE QUOTE CUSTOMERS BY REGION (UPDATED)
print("\n" + "="*80)
print("SINGLE QUOTE CUSTOMERS BY REGION")
print("="*80)

# Count quotes per customer
quote_counts = df_quotes.groupby('numero_compte').size().reset_index(name='quote_count')
single_quote_ids = quote_counts[quote_counts['quote_count'] == 1]['numero_compte'].tolist()

# Single quote customers by region
single_quote_by_region = df_quotes[df_quotes['numero_compte'].isin(single_quote_ids)] \
    .groupby('nom_region')['numero_compte'].nunique() \
    .reset_index(name='single_quote_customers')

# Total customers by region
total_by_region = df_quotes.groupby('nom_region')['numero_compte'].nunique().reset_index(name='total_customers')

# Merge
single_region_stats = single_quote_by_region.merge(total_by_region, on='nom_region', how='right').fillna(0)
single_region_stats['pct_of_region'] = (single_region_stats['single_quote_customers'] / single_region_stats['total_customers'] * 100).round(1)

# Sort and display
single_region_stats = single_region_stats.sort_values('single_quote_customers', ascending=False)
print(single_region_stats[['nom_region', 'single_quote_customers', 'pct_of_region']].to_string(index=False))

print("\nWarm regions have more single-quote customers.")


SINGLE QUOTE CUSTOMERS BY REGION
          nom_region  single_quote_customers  pct_of_region
           Normandie                    7649           69.5
Auvergne-Rhône-Alpes                    3951           69.2
       Île-de-France                    3178           67.2
     Hauts-de-France                    2613           71.8
                 Sud                     467           72.5

Warm regions have more single-quote customers.


In [5]:
# First, let's check what product columns exist in the original quote-level data
print("="*80)
print("QUOTE-LEVEL PRODUCT COLUMN ANALYSIS")
print("="*80)

# Load the raw data if not already loaded
if 'df_clean' not in locals():
    df_clean = pd.read_csv('cleaned_quote_data.csv')
    df_clean['dt_creation_devis'] = pd.to_datetime(df_clean['dt_creation_devis'])
    print("Loaded df_clean")

# Check for product-related columns
product_cols_quote = [col for col in df_clean.columns if any(x in col.lower() for x in ['famille', 'type', 'equip', 'marque'])]
print(f"\nProduct-related columns in quote-level data:")
for col in product_cols_quote:
    print(f"  - {col}")

# Focus on the three key product columns
key_product_cols = ['famille_equipement_produit', 
                    'famille_equipement_produit_principal',
                    'regroup_famille_equipement_produit_principal']

print(f"\n" + "="*80)
print("ANALYSIS OF 3 PRODUCT COLUMNS AT QUOTE LEVEL")
print("="*80)

# Check which columns exist
existing_cols = []
for col in key_product_cols:
    if col in df_clean.columns:
        existing_cols.append(col)
        print(f"\n✅ {col} exists")
    else:
        print(f"\n❌ {col} MISSING")

# For existing columns, analyze their content
for col in existing_cols:
    print(f"\n" + "="*80)
    print(f"COLUMN: {col}")
    print("="*80)
    
    # Value counts
    print(f"\nValue counts (top 15):")
    print(df_clean[col].value_counts().head(15))
    
    # Missing values
    missing = df_clean[col].isna().sum()
    print(f"\nMissing values: {missing:,} ({missing/len(df_clean)*100:.1f}%)")
    
    # Unique values
    print(f"Unique values: {df_clean[col].nunique()}")

# If multiple columns exist, create a cross-tabulation
if len(existing_cols) >= 2:
    print("\n" + "="*80)
    print("CROSS-TABULATIONS BETWEEN COLUMNS")
    print("="*80)
    
    # Compare each pair
    for i, col1 in enumerate(existing_cols):
        for col2 in existing_cols[i+1:]:
            print(f"\n{col1} vs {col2}:")
            print("-" * 60)
            
            # Create cross-tabulation
            cross = pd.crosstab(df_clean[col1], df_clean[col2], normalize='index') * 100
            print(cross.round(1).head(20))

# Create a sample of quotes showing all three columns side by side
print("\n" + "="*80)
print("SAMPLE QUOTES: ALL THREE COLUMNS SIDE BY SIDE")
print("="*80)

# Create a sample DataFrame with all columns that exist
sample_cols = ['id_devis', 'numero_compte'] + existing_cols
sample_df = df_clean[sample_cols].head(30)

print("\nSample of 30 quotes:")
print(sample_df.to_string())

# Count how many quotes have different values across columns
if len(existing_cols) >= 2:
    print("\n" + "="*80)
    print("MISMATCH ANALYSIS")
    print("="*80)
    
    # Check where values differ between columns
    for i, col1 in enumerate(existing_cols):
        for col2 in existing_cols[i+1:]:
            mismatch = df_clean[df_clean[col1] != df_clean[col2]]
            print(f"\n{col1} vs {col2}:")
            print(f"  Total quotes: {len(df_clean):,}")
            print(f"  Mismatched quotes: {len(mismatch):,} ({len(mismatch)/len(df_clean)*100:.1f}%)")
            
            # Show sample mismatches
            if len(mismatch) > 0:
                print(f"\n  Sample mismatches (first 10):")
                sample_mismatch = mismatch[[col1, col2, 'id_devis', 'numero_compte']].head(10)
                print(sample_mismatch.to_string())

# If only one column exists, check if others can be derived
if len(existing_cols) == 1:
    print("\n" + "="*80)
    print("DERIVATION POSSIBILITIES")
    print("="*80)
    
    # Show examples of the existing column
    print(f"\nUnique values in {existing_cols[0]}:")
    print(df_clean[existing_cols[0]].value_counts().head(20))

QUOTE-LEVEL PRODUCT COLUMN ANALYSIS

Product-related columns in quote-level data:
  - type_devis
  - famille_equipement_produit
  - type_equipement_produit
  - regroup_famille_equipement_produit
  - marque_produit
  - famille_equipement_produit_principal
  - regroup_famille_equipement_produit_principal

ANALYSIS OF 3 PRODUCT COLUMNS AT QUOTE LEVEL

✅ famille_equipement_produit exists

✅ famille_equipement_produit_principal exists

✅ regroup_famille_equipement_produit_principal exists

COLUMN: famille_equipement_produit

Value counts (top 15):
famille_equipement_produit
Chaudière                           10733
Poêle                                8021
Climatisation                        7348
Pompe à chaleur                      6808
ECS : Chauffe-eau ou adoucisseur     1731
Photovoltaïque                       1141
Autres                                405
Appareil hybride                      214
Plomberie Sanitaire                   193
Produit VMC                           175
Emet

In [6]:
print("="*80)
print("DEEP DIVE: famille_equipement_produit vs famille_equipement_produit_principal (QUOTE LEVEL)")
print("="*80)

# First, let's verify we're working with quote-level data
if 'df_clean' not in locals():
    df_clean = pd.read_csv('cleaned_quote_data.csv')
    df_clean['dt_creation_devis'] = pd.to_datetime(df_clean['dt_creation_devis'])
    print("Loaded df_clean")

# Create a copy for analysis
df_analysis = df_clean.copy()

# Identify mismatched rows (both columns have values, but different)
mismatch_mask = (df_analysis['famille_equipement_produit'] != df_analysis['famille_equipement_produit_principal']) & \
                (df_analysis['famille_equipement_produit'].notna()) & \
                (df_analysis['famille_equipement_produit_principal'].notna())

mismatch_df = df_analysis[mismatch_mask]
total_mismatch = len(mismatch_df)
total_quotes = len(df_analysis)

print(f"\n1. OVERVIEW")
print("-" * 60)
print(f"Total quotes: {total_quotes:,}")
print(f"Quotes with BOTH columns populated: {(df_analysis['famille_equipement_produit'].notna() & df_analysis['famille_equipement_produit_principal'].notna()).sum():,}")
print(f"Quotes with mismatches (different values): {total_mismatch:,} ({total_mismatch/total_quotes*100:.2f}%)")
print(f"Quotes with same value: {(df_analysis['famille_equipement_produit'] == df_analysis['famille_equipement_produit_principal']).sum():,}")

# Analyze the mismatches
print("\n" + "="*80)
print("2. MISMATCH PATTERNS - WHAT OLD VALUE GOES TO WHAT NEW VALUE?")
print("="*80)

# Create a crosstab of mismatches
mismatch_cross = pd.crosstab(
    mismatch_df['famille_equipement_produit'],
    mismatch_df['famille_equipement_produit_principal']
)

print("\nMismatch matrix (rows = old column, columns = new column):")
print(mismatch_cross)

# Show the most frequent mismatch pairs
print("\n" + "="*80)
print("3. MOST FREQUENT MISMATCH PAIRS")
print("="*80)

mismatch_pairs = mismatch_df.groupby(['famille_equipement_produit', 'famille_equipement_produit_principal']).size().reset_index(name='count')
mismatch_pairs = mismatch_pairs.sort_values('count', ascending=False)

print("\nAll mismatch pairs:")
for i, row in mismatch_pairs.iterrows():
    print(f"  {row['famille_equipement_produit']:40} → {row['famille_equipement_produit_principal']:40} : {row['count']:,} quotes")

# Analyze where the old column has values that don't exist in new column
print("\n" + "="*80)
print("4. VALUES UNIQUE TO EACH COLUMN")
print("="*80)

old_unique = set(df_analysis['famille_equipement_produit'].dropna().unique())
new_unique = set(df_analysis['famille_equipement_produit_principal'].dropna().unique())

only_in_old = old_unique - new_unique
print(f"\nValues only in old column: {only_in_old}")
if only_in_old:
    for val in only_in_old:
        count = df_analysis[df_analysis['famille_equipement_produit'] == val].shape[0]
        # Check what they map to in new column
        mapping = df_analysis[df_analysis['famille_equipement_produit'] == val]['famille_equipement_produit_principal'].value_counts()
        print(f"  {val}: {count:,} quotes → maps to: {dict(mapping.head(3))}")

only_in_new = new_unique - old_unique
print(f"\nValues only in new column: {only_in_new}")
if only_in_new:
    for val in only_in_new:
        count = df_analysis[df_analysis['famille_equipement_produit_principal'] == val].shape[0]
        # Check what they map to from old column
        mapping = df_analysis[df_analysis['famille_equipement_produit_principal'] == val]['famille_equipement_produit'].value_counts()
        print(f"  {val}: {count:,} quotes ← mapped from: {dict(mapping.head(3))}")

# Analyze the mapping for each major category
print("\n" + "="*80)
print("5. DETAILED MAPPING BY MAJOR CATEGORY")
print("="*80)

# Focus on major categories
major_categories = ['Chaudière', 'Poêle', 'Climatisation', 'Pompe à chaleur', 'Appareil hybride', 'Autres']

for category in major_categories:
    print(f"\n{category}:")
    print("-" * 40)
    
    # Subset where old column equals this category
    subset_old = df_analysis[df_analysis['famille_equipement_produit'] == category]
    if len(subset_old) > 0:
        print(f"  In old column: {len(subset_old):,} quotes")
        new_values = subset_old['famille_equipement_produit_principal'].value_counts()
        print(f"  New column distribution:")
        for val, count in new_values.items():
            print(f"    {val}: {count:,} ({count/len(subset_old)*100:.1f}%)")
    
    # Subset where new column equals this category
    subset_new = df_analysis[df_analysis['famille_equipement_produit_principal'] == category]
    if len(subset_new) > 0:
        print(f"\n  In new column: {len(subset_new):,} quotes")
        old_values = subset_new['famille_equipement_produit'].value_counts()
        print(f"  Old column distribution:")
        for val, count in old_values.items():
            print(f"    {val}: {count:,} ({count/len(subset_new)*100:.1f}%)")

# Analyze special cases - hybrid and other
print("\n" + "="*80)
print("6. SPECIAL CASES ANALYSIS")
print("="*80)

# Appareil hybride analysis
hybrid = df_analysis[df_analysis['famille_equipement_produit'] == 'Appareil hybride']
print(f"\n'Appareil hybride' in old column: {len(hybrid):,} quotes")
if len(hybrid) > 0:
    print("  Distribution in new column:")
    new_dist = hybrid['famille_equipement_produit_principal'].value_counts()
    for val, count in new_dist.items():
        print(f"    {val}: {count:,} ({count/len(hybrid)*100:.1f}%)")

# ECS (water heater) analysis
ecs = df_analysis[df_analysis['famille_equipement_produit'] == 'ECS : Chauffe-eau ou adoucisseur']
print(f"\n'ECS : Chauffe-eau ou adoucisseur' in old column: {len(ecs):,} quotes")
if len(ecs) > 0:
    print("  Distribution in new column:")
    new_dist = ecs['famille_equipement_produit_principal'].value_counts()
    for val, count in new_dist.items():
        print(f"    {val}: {count:,} ({count/len(ecs)*100:.1f}%)")

# Photovoltaïque analysis
solar = df_analysis[df_analysis['famille_equipement_produit'] == 'Photovoltaïque']
print(f"\n'Photovoltaïque' in old column: {len(solar):,} quotes")
if len(solar) > 0:
    print("  Distribution in new column:")
    new_dist = solar['famille_equipement_produit_principal'].value_counts()
    for val, count in new_dist.items():
        print(f"    {val}: {count:,} ({count/len(solar)*100:.1f}%)")

# Analyze the impact on conversion rates
print("\n" + "="*80)
print("7. IMPACT ON CONVERSION RATES")
print("="*80)

# Compare conversion rates for mismatched vs matched quotes
matched_quotes = df_analysis[(df_analysis['famille_equipement_produit'] == df_analysis['famille_equipement_produit_principal']) & 
                              (df_analysis['famille_equipement_produit'].notna()) & 
                              (df_analysis['famille_equipement_produit_principal'].notna())]
matched_conv = matched_quotes['fg_devis_accepte'].mean()

mismatch_conv = mismatch_df['fg_devis_accepte'].mean()

print(f"\nConversion rate for matched quotes: {matched_conv:.2%} (n={len(matched_quotes):,})")
print(f"Conversion rate for mismatched quotes: {mismatch_conv:.2%} (n={len(mismatch_df):,})")
print(f"Difference: {(mismatch_conv - matched_conv)*100:.2f} points")

# Check if mismatches affect conversion differently by category
print("\n" + "="*80)
print("8. CONVERSION IMPACT BY CATEGORY")
print("="*80)

for category in ['Chaudière', 'Poêle', 'Climatisation', 'Pompe à chaleur']:
    cat_matched = matched_quotes[matched_quotes['famille_equipement_produit'] == category]
    cat_mismatch = mismatch_df[mismatch_df['famille_equipement_produit'] == category]
    
    if len(cat_matched) > 0 and len(cat_mismatch) > 0:
        print(f"\n{category}:")
        print(f"  Matched conversion: {cat_matched['fg_devis_accepte'].mean():.2%} (n={len(cat_matched):,})")
        print(f"  Mismatched conversion: {cat_mismatch['fg_devis_accepte'].mean():.2%} (n={len(cat_mismatch):,})")
        print(f"  Difference: {(cat_mismatch['fg_devis_accepte'].mean() - cat_matched['fg_devis_accepte'].mean())*100:.2f} points")

# Summary recommendation
print("\n" + "="*80)
print("9. SUMMARY & RECOMMENDATION")
print("="*80)

print(f"\nBased on {total_mismatch:,} mismatched quotes ({total_mismatch/total_quotes*100:.2f}% of total):")
print(f"  - Most mismatches are from the old column 'Appareil hybride' and 'Autres'")
print(f"  - The new column has more granular categories (14 vs 12)")
print(f"  - Conversion rates differ by {abs((mismatch_conv - matched_conv)*100):.2f} points between matched and mismatched quotes")

print("\n📌 RECOMMENDATION:")
print("  For customer-level aggregation, use:")
print("    - 'regroup_famille_equipement_produit_principal' for clean English grouping")
print("    - 'famille_equipement_produit_principal' for detailed French analysis")
print("  The old 'famille_equipement_produit' column has mismatches that may affect analysis")

DEEP DIVE: famille_equipement_produit vs famille_equipement_produit_principal (QUOTE LEVEL)

1. OVERVIEW
------------------------------------------------------------
Total quotes: 37,008
Quotes with BOTH columns populated: 36,752
Quotes with mismatches (different values): 3,382 (9.14%)
Quotes with same value: 33,370

2. MISMATCH PATTERNS - WHAT OLD VALUE GOES TO WHAT NEW VALUE?

Mismatch matrix (rows = old column, columns = new column):
famille_equipement_produit_principal  Accessoire de pose  Appareil hybride  \
famille_equipement_produit                                                   
Appareil hybride                                       0                 0   
Autres                                                 1                 1   
Chaudière                                              7                23   
Climatisation                                          1                 0   
ECS : Chauffe-eau ou adoucisseur                      56                 0   
Emetteur de c

In [7]:
# Check Atlantic quotes in the raw data
print("="*80)
print("ATLANTIC QUOTES - RAW PRODUCT CLASSIFICATION")
print("="*80)

# Get all Atlantic quotes
atlantic_quotes = df_clean[df_clean['marque_produit'] == 'ATLANTIC']

print(f"\nTotal Atlantic quotes: {len(atlantic_quotes):,}")

# Check the new product column (clean English grouping)
print("\n1. NEW PRODUCT COLUMN (regroup_famille_equipement_produit_principal):")
print(atlantic_quotes['regroup_famille_equipement_produit_principal'].value_counts())

# Check the detailed French column
print("\n2. DETAILED FRENCH COLUMN (famille_equipement_produit_principal):")
print(atlantic_quotes['famille_equipement_produit_principal'].value_counts())

# Check the old product column for comparison
print("\n3. OLD PRODUCT COLUMN (famille_equipement_produit):")
print(atlantic_quotes['famille_equipement_produit'].value_counts())

# Check if any Atlantic quotes are marked as STOVE in the new column
stove_count = (atlantic_quotes['regroup_famille_equipement_produit_principal'] == 'STOVE').sum()
print(f"\n❌ Atlantic quotes marked as STOVE in new column: {stove_count}")

# Show sample of any misclassified quotes
if stove_count > 0:
    print("\nSample of misclassified Atlantic quotes:")
    misclassified = atlantic_quotes[atlantic_quotes['regroup_famille_equipement_produit_principal'] == 'STOVE']
    print(misclassified[['id_devis', 'marque_produit', 'regroup_famille_equipement_produit_principal', 
                         'famille_equipement_produit_principal', 'famille_equipement_produit']].head(10))

ATLANTIC QUOTES - RAW PRODUCT CLASSIFICATION

Total Atlantic quotes: 6,180

1. NEW PRODUCT COLUMN (regroup_famille_equipement_produit_principal):
regroup_famille_equipement_produit_principal
HEAT_PUMP          2713
AIR_CONDITIONER    1632
OTHER              1031
BOILER_GAS          804
Name: count, dtype: int64

2. DETAILED FRENCH COLUMN (famille_equipement_produit_principal):
famille_equipement_produit_principal
Pompe à chaleur                     2713
Climatisation                       1632
Chaudière                            804
ECS : Chauffe-eau ou adoucisseur     779
Appareil hybride                     104
Produit VMC                          103
Accessoire de pose                    34
Emetteur de chauffage  ou chappe      11
Name: count, dtype: int64

3. OLD PRODUCT COLUMN (famille_equipement_produit):
famille_equipement_produit
Pompe à chaleur                     2560
Climatisation                       1587
Chaudière                           1013
ECS : Chauffe-eau ou adouc